![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 03: Big Data)**

**LabClass M03A: Parquet, JSON, pandas, and NumPy ETL**

---

- Materials in this module have been developed to support practical learning in modern data science, big data processing, and applied analytics.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find an issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

## LabClass M03A: Parquet, JSON, pandas, and NumPy ETL

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This lab class is the A stream practical demonstration for Module 03. It works with public bank parquet and nested JSON data, and it is one component of the LabClasses A tutor-pack sequence.</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter with pandas, NumPy, and the listed optional packages.</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>A parquet inspection workflow, filtered JSON outputs, NumPy feature preparation, and a bounded distance matrix.</td>
</tr>
<tr>
<td align="left">Related assessment</td>
<td>General practical skill development. Not directly assessed.</td>
</tr>
</tbody>
</table>

</div>

---


**Table of Contents**

- [1. Overview and Learning Goals](#1.-overview-and-learning-goals)
- [2. Setup and Data Files](#2.-setup-and-data-files)
- [3. Read the Parquet File](#3.-read-the-parquet-file)
- [4. Inspect and Filter Dictionaries](#4.-inspect-and-filter-dictionaries)
- [5. Build a Small Parser](#5.-build-a-small-parser)
- [6. NumPy ETL and Feature Preparation](#6.-numpy-etl-and-feature-preparation)
- [7. Nested JSON and Distance Matrix](#7.-nested-json-and-distance-matrix)
- [8. Student Tasks](#8.-student-tasks)
- [9. Checks](#9.-checks)
- [10. Reflection and References](#10.-reflection-and-references)


<a id="1-overview-and-learning-goals"></a>
### 1. Overview and Learning Goals

This practical works with the public `bank.parquet` file and a small nested JSON file. The goal is to move between parquet, pandas DataFrames, dictionaries, JSON outputs, NumPy arrays, and model-ready numeric features.

By the end of this lab, students should be able to:

1. load a parquet file with `pyarrow` and pandas;
2. inspect dictionary-style column storage;
3. filter records and save a JSON output;
4. use NumPy broadcasting for a simple ETL step;
5. separate numeric and categorical features; and
6. compute a small Euclidean distance matrix.


<a id="2-setup-and-data-files"></a>
### 2. Setup and Data Files

This notebook uses `bank.parquet` and `elevations.json`.

If `pyarrow` is not available in your notebook environment, install it in a separate setup cell with `%pip install pyarrow`, then restart or rerun the relevant cells. Many Colab runtimes already include it.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import json
import sys
import tempfile

import numpy as np
import pandas as pd

PUBLIC_DATA_BASE_URL = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
EXECUTION_MODE = "online"  # Use "online" for Google Colab; use "local" for a cloned SIT742 repository.

OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="sit742_m03a_output_"))
required_files = ['bank.parquet', 'elevations.json']


def download_public_data(required_files):
    data_dir = Path(tempfile.mkdtemp(prefix="sit742_m03a_data_"))
    downloaded_paths = {}
    for filename in required_files:
        local_file = data_dir / filename
        urlretrieve(f"{PUBLIC_DATA_BASE_URL}/{filename}", local_file)
        downloaded_paths[filename] = local_file
    return data_dir, downloaded_paths


def find_local_data_dir(required_files):
    candidates = [
        Path.cwd() / "Jupyter" / "data",
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd().parent.parent / "Jupyter" / "data",
    ]
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required_files):
            return candidate
    searched = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError("Could not find the SIT742 public data folder. Searched:\n" + searched)


if EXECUTION_MODE == "online":
    DATA_DIR, data_paths_by_file = download_public_data(required_files)
elif EXECUTION_MODE == "local":
    DATA_DIR = find_local_data_dir(required_files)
    data_paths_by_file = {filename: DATA_DIR / filename for filename in required_files}
else:
    raise ValueError('EXECUTION_MODE must be "online" or "local".')

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)


In [ ]:
try:
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError("This notebook requires pyarrow. In a notebook cell, run `%pip install pyarrow` and then rerun setup.") from exc

bank_path = data_paths_by_file["bank.parquet"]
elevations_path = data_paths_by_file["elevations.json"]
print("Bank parquet:", bank_path)
print("Elevations JSON:", elevations_path)


<a id="3-read-parquet"></a>
### 3. Read the Parquet File

Parquet stores typed columns efficiently. Use `pyarrow` for schema and metadata inspection, then use pandas for table analysis.


In [ ]:
parquet_file = pq.ParquetFile(bank_path)
print(parquet_file.schema)
print("Rows:", parquet_file.metadata.num_rows)
print("Columns:", parquet_file.metadata.num_columns)

table = parquet_file.read()
bank_dict = table.to_pydict()
bank_df = pd.read_parquet(bank_path, engine="pyarrow")
bank_df.head()


<a id="4-inspect-and-filter-dictionaries"></a>
### 4. Inspect and Filter Dictionaries

The dictionary representation stores one list of values per column. Filter rows where `age` is greater than 50 and less than 70, then save the filtered records as JSON.


In [ ]:
for key, values in bank_dict.items():
    unique_count = pd.Series(values).nunique(dropna=False)
    print(f"{key}: {len(values)} values, {unique_count} unique values")

age_mask = (bank_df["age"] > 50) & (bank_df["age"] < 70)
bank_filtered = bank_df.loc[age_mask].reset_index(drop=True)

bank_json_path = OUTPUT_DIR / "bank_age_50_to_70.json"
bank_filtered.to_json(bank_json_path, orient="records", indent=2)
bank_json = pd.read_json(bank_json_path)

print("Filtered rows:", len(bank_json))
bank_json.head()


<a id="5-build-a-small-parser"></a>
### 5. Build a Small Parser

Wrap the parquet-reading and row-filtering logic in a function so the same process can be rerun with different numeric conditions.


In [ ]:
def parquet_filter_to_json(source_path, output_path, filter_column, min_value, max_value):
    frame = pd.read_parquet(source_path, engine="pyarrow")
    mask = (frame[filter_column] > min_value) & (frame[filter_column] < max_value)
    filtered = frame.loc[mask].reset_index(drop=True)
    filtered.to_json(output_path, orient="records", indent=2)
    return filtered


newbank_path = OUTPUT_DIR / "newbank.json"
df_newbank = parquet_filter_to_json(bank_path, newbank_path, "age", 30, 35)
print("newbank rows:", len(df_newbank))
df_newbank.head()


<a id="6-numpy-etl-and-feature-preparation"></a>
### 6. NumPy ETL and Feature Preparation

Use NumPy broadcasting to add 10 to each age value, then prepare numeric and one-hot encoded categorical features.


In [ ]:
np_df_age = df_newbank[["age"]].to_numpy()
np_df_age_new = np_df_age + 10

df_newbank_new = df_newbank.copy()
df_newbank_new.insert(0, "new_age", np_df_age_new.ravel())

num_col = bank_df.select_dtypes(include=np.number).columns
cat_col = bank_df.select_dtypes(exclude=np.number).columns

df_num = bank_df[num_col]
df_cat = bank_df[cat_col]
df_onehot = pd.get_dummies(df_cat, dtype=int)
df_all = pd.concat([df_num.reset_index(drop=True), df_onehot.reset_index(drop=True)], axis=1)

print("Age array shape:", np_df_age.shape)
print("Prepared feature shape:", df_all.shape)
df_newbank_new.head()


<a id="7-nested-json-and-distance"></a>
### 7. Nested JSON and Distance Matrix

Load the elevation JSON file as an example of nested web-service-style data, then compute nearest-neighbour indices from the prepared numeric feature table.


In [ ]:
with elevations_path.open("r", encoding="utf-8") as fp:
    elevations_payload = json.load(fp)
if isinstance(elevations_payload, str):
    elevations_payload = json.loads(elevations_payload)

elevation_records = [
    {
        "lat": item["location"]["lat"],
        "lng": item["location"]["lng"],
        "elevation": item["elevation"],
        "resolution": item["resolution"],
    }
    for item in elevations_payload["results"]
]
elevations_df = pd.DataFrame(elevation_records)
elevations_df


In [ ]:
def distance_to_one(row1, row2):
    return np.sqrt(np.sum(np.square(row1 - row2)))


def distance_to_many(row1, rows):
    return np.sqrt(np.sum(np.square(rows - row1), axis=1))


def nearest_neighbour_summary(frame, sample_size=100):
    sample = frame.head(sample_size).to_numpy(dtype=float)
    distance_matrix = np.vstack([distance_to_many(row, sample) for row in sample])
    nearest = []
    for index, distances in enumerate(distance_matrix):
        order = np.argsort(distances)
        nearest.append(order[1] if len(order) > 1 else order[0])
    return distance_matrix, pd.DataFrame({"index": range(len(nearest)), "most_similar_index": nearest})


In [ ]:
distance_matrix, nearest = nearest_neighbour_summary(df_all, sample_size=100)
print("Distance matrix shape:", distance_matrix.shape)
nearest.head(10)


<a id="8-student-tasks"></a>
### 8. Student Tasks

1. Change the bank age filter to a different five-year range and report the row count.
2. Compare the number of numeric and categorical columns before one-hot encoding.
3. Compute the most common `job` value in the filtered dataset.
4. Explain why one-hot encoding increases the number of columns.
5. Try `sample_size=50` in the nearest-neighbour function and compare the matrix shape.


In [ ]:
# Task workspace.
age_40_45 = parquet_filter_to_json(bank_path, OUTPUT_DIR / "bank_age_40_to_45.json", "age", 40, 45)
print("Rows for age 40 to 45:", len(age_40_45))
print("Most common filtered job:", bank_filtered["job"].mode().iloc[0])


<a id="9-checks"></a>
### 9. Checks


In [ ]:
assert bank_df.shape[0] == 11162
assert "age" in bank_df.columns
assert bank_json_path.exists()
assert newbank_path.exists()
assert df_all.shape[0] == bank_df.shape[0]
assert distance_matrix.shape == (100, 100)
assert len(elevations_df) == 2

print("M03PracClass-A checks passed.")


<a id="10-reflection-and-references"></a>
### 10. Reflection and References

Reflection prompts:

1. When would parquet be more useful than CSV?
2. What information is easier to inspect in a dictionary than in a DataFrame?
3. Why should generated JSON files be written to an output folder?

References:

- pandas documentation: `read_parquet`, `get_dummies`, `to_json`
- NumPy documentation: broadcasting and array operations
- Apache Parquet documentation
